# Gold — Transações por caixa por dia

Desenvolvido por Ygor Moraes

Este notebook cria a Gold de volume de transações por caixa por dia.

Fonte:
- Silver `physical_vendas_caixa`

Granularidade:
- 1 linha por loja, caixa e data de venda

Destino ADLS:
- `gold/physical_vendas_caixa/transacoes_caixa_dia`

Destino SQL Server:
- `squad3.gold_physical_vendas_caixa_transacoes_caixa_dia`

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    sum as spark_sum,
    to_date
)

SILVER_TABLE = "physical_vendas_caixa"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

GOLD_DOMAIN = "physical_vendas_caixa"
GOLD_KPI = "transacoes_caixa_dia"

GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_DOMAIN}/{GOLD_KPI}"

FINAL_TABLE_NAME = f"gold_{GOLD_DOMAIN}_{GOLD_KPI}"
FINAL_TABLE = f"{TARGET_SCHEMA}.{FINAL_TABLE_NAME}"

GOLD_WRITE_MODE = "overwrite"

SILVER_REQUIRED_COLUMNS = [
    "id_transacao",
    "id_loja",
    "id_caixa",
    "dt_venda",
    "valor_total_venda",
    "ano",
    "mes"
]

GOLD_KEY_COLUMNS = [
    "id_loja",
    "id_caixa",
    "data_venda"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê a Silver de vendas físicas e valida colunas obrigatórias.

df_vendas = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

validate_required_columns(df_vendas, SILVER_REQUIRED_COLUMNS)

total_vendas = df_vendas.count()

print("Silver de physical_vendas_caixa lida com sucesso.")
print(f"Total de registros: {total_vendas}")

df_vendas.printSchema()

display(df_vendas.limit(10))

In [0]:
# Cria a data da venda a partir do timestamp.

df_vendas_dia = (
    df_vendas
    .withColumn(
        "data_venda",
        to_date(col("dt_venda"))
    )
)

print("Data da venda criada.")

display(
    df_vendas_dia
    .select(
        "id_transacao",
        "id_loja",
        "id_caixa",
        "dt_venda",
        "data_venda",
        "valor_total_venda"
    )
    .limit(20)
)

In [0]:
# Calcula o número de transações por loja, caixa e dia.

df_transacoes_caixa_dia = (
    df_vendas_dia
    .groupBy(
        "id_loja",
        "id_caixa",
        "data_venda"
    )
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total")
    )
    .withColumn(
        "ano",
        col("data_venda").cast("date").substr(1, 4).cast("int")
    )
    .withColumn(
        "mes",
        col("data_venda").cast("date").substr(6, 2).cast("int")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print("Transações por caixa por dia calculadas.")
print(f"Total de linhas agregadas: {df_transacoes_caixa_dia.count()}")

display(
    df_transacoes_caixa_dia
    .orderBy("id_loja", "id_caixa", "data_venda")
    .limit(30)
)

In [0]:
# Organiza o schema final da Gold.

df_gold_final = (
    df_transacoes_caixa_dia
    .select(
        "id_loja",
        "id_caixa",
        "data_venda",
        "ano",
        "mes",
        "qtd_transacoes",
        "receita_total",
        "gold_processed_at"
    )
)

print("Schema final da Gold definido.")

df_gold_final.printSchema()

display(
    df_gold_final
    .orderBy("id_loja", "id_caixa", "data_venda")
    .limit(30)
)

In [0]:
# Valida volume, duplicidade e consistência das transações.

total_gold = df_gold_final.count()

duplicados_gold = (
    df_gold_final
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_transacoes_gold = (
    df_gold_final
    .agg(spark_sum("qtd_transacoes").alias("qtd_transacoes"))
    .collect()[0]["qtd_transacoes"]
)

receita_silver = (
    df_vendas
    .agg(spark_sum("valor_total_venda").alias("receita_total"))
    .collect()[0]["receita_total"]
)

receita_gold = (
    df_gold_final
    .agg(spark_sum("receita_total").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Total de linhas Gold: {total_gold}")
print(f"Chaves duplicadas na Gold: {duplicados_gold}")
print(f"Total transações Silver: {total_vendas}")
print(f"Total transações Gold: {total_transacoes_gold}")
print(f"Receita total Silver: {receita_silver}")
print(f"Receita total Gold: {receita_gold}")

if duplicados_gold > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold.")

if total_transacoes_gold != total_vendas:
    raise Exception("Erro: total de transações da Gold não bate com a Silver.")

if receita_silver != receita_gold:
    raise Exception("Erro: receita total da Gold não bate com a Silver.")

print("Validação OK: Gold sem duplicidade e transações consistentes.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(GOLD_PATH)
)

print(f"Gold Delta gravada com sucesso em: {GOLD_PATH}")

In [0]:
# Lê a Gold Delta gravada para validar persistência.

df_gold_delta = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Gold Delta lida com sucesso.")
print(f"Total de linhas gravadas: {df_gold_delta.count()}")

df_gold_delta.printSchema()

display(
    df_gold_delta
    .orderBy("id_loja", "id_caixa", "data_venda")
    .limit(30)
)

In [0]:
# Valida volume, duplicidade e consistência da Gold gravada.

total_gold_memoria = df_gold_final.count()
total_gold_delta = df_gold_delta.count()

duplicados_gold_delta = (
    df_gold_delta
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_transacoes_delta = (
    df_gold_delta
    .agg(spark_sum("qtd_transacoes").alias("qtd_transacoes"))
    .collect()[0]["qtd_transacoes"]
)

receita_gold_delta = (
    df_gold_delta
    .agg(spark_sum("receita_total").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Total Gold em memória: {total_gold_memoria}")
print(f"Total Gold Delta: {total_gold_delta}")
print(f"Chaves duplicadas na Gold Delta: {duplicados_gold_delta}")
print(f"Total transações Silver: {total_vendas}")
print(f"Total transações Gold Delta: {total_transacoes_delta}")
print(f"Receita total Silver: {receita_silver}")
print(f"Receita total Gold Delta: {receita_gold_delta}")

if total_gold_memoria != total_gold_delta:
    raise Exception("Erro: quantidade gravada diferente da Gold em memória.")

if duplicados_gold_delta > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold Delta.")

if total_transacoes_delta != total_vendas:
    raise Exception("Erro: total de transações da Gold Delta não bate com a Silver.")

if receita_silver != receita_gold_delta:
    raise Exception("Erro: receita total da Gold Delta não bate com a Silver.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_delta
    .select(
        col("id_loja").cast("int").alias("id_loja"),
        col("id_caixa").cast("int").alias("id_caixa"),
        col("data_venda").cast("date").alias("data_venda"),
        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes"),
        col("qtd_transacoes").cast("int").alias("qtd_transacoes"),
        col("receita_total").cast("decimal(18,2)").alias("receita_total"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")

df_gold_sql.printSchema()

display(
    df_gold_sql
    .orderBy("id_loja", "id_caixa", "data_venda")
    .limit(30)
)

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final gravada no SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_sql = df_final.count()

duplicados_sql = (
    df_final
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_transacoes_sql = (
    df_final
    .agg(spark_sum("qtd_transacoes").alias("qtd_transacoes"))
    .collect()[0]["qtd_transacoes"]
)

receita_sql = (
    df_final
    .agg(spark_sum("receita_total").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Total Gold Delta: {total_gold_delta}")
print(f"Total tabela final SQL Server: {total_sql}")
print(f"Chaves duplicadas SQL Server: {duplicados_sql}")
print(f"Total transações SQL Server: {total_transacoes_sql}")
print(f"Receita Gold Delta: {receita_gold_delta}")
print(f"Receita SQL Server: {receita_sql}")

if total_sql != total_gold_delta:
    raise Exception("Erro: quantidade no SQL Server diferente da Gold Delta.")

if duplicados_sql > 0:
    raise Exception("Erro: existem chaves duplicadas no SQL Server.")

if total_transacoes_sql != total_vendas:
    raise Exception("Erro: total de transações no SQL Server não bate com a Silver.")

if receita_sql != receita_gold_delta:
    raise Exception("Erro: receita total no SQL Server não bate com a Gold Delta.")

print("Validação OK: tabela final SQL Server gravada corretamente.")